In [103]:
# Notebook 01: Merge and Feature Build
#
# Purpose:
# - Load cleaned Alabama_SoS and USDA county-level datasets
# - Inspect columns and merge keys
# - Merge datasets into one county-level modeling table
# - Create derived features for PCA
# - Export the merged dataset

In [104]:
import pandas as pd
import numpy as np
from pathlib import Path

In [105]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "cleaned"

AL_SOS_DIR = DATA_DIR / "Alabama_SoS"
USDA_DIR = DATA_DIR / "USDA"

ballots_file = AL_SOS_DIR / "2024 General Election Total Ballots Cast.csv"
race_file_2024 = AL_SOS_DIR / "2024 General Election Participation by Race.csv"
race_file_2020 = AL_SOS_DIR / "2020 General Election Participation by Race.csv"
age_file = AL_SOS_DIR / "2024 General Election Participation by Age.csv"
gender_file = AL_SOS_DIR / "2024 General Election Participation by Gender.csv"


education_file = USDA_DIR / "Educational attainment for adults age 25 and older for the United States, States, and counties, 1970–2023.csv"
population_file = USDA_DIR / "Population estimates for the United States, States, and counties, 2020–23.csv"
poverty_file = USDA_DIR / "Poverty estimates for the United States, States, and counties, 2023.csv"
income_file = USDA_DIR / "Unemployment and median household income for the United States, States, and counties, 2000–23.csv"

broadband_file = DATA_DIR / "CLEANED alabama-broadband-access-county-sort.csv"

In [106]:
ballots = pd.read_csv(ballots_file)
race_2024 = pd.read_csv(race_file_2024)
race_2020 = pd.read_csv(race_file_2020)
age = pd.read_csv(age_file)
gender = pd.read_csv(gender_file)

education = pd.read_csv(education_file)
population = pd.read_csv(population_file)
poverty = pd.read_csv(poverty_file)
income = pd.read_csv(income_file)

broadband = pd.read_csv(broadband_file)

In [107]:
datasets = {
    "ballots": ballots,
    "race24": race_2024,
    "race20": race_2020,
    "age": age,
    "gender": gender,
    "education": education,
    "population": population,
    "poverty": poverty,
    "income": income,
    "broadband": broadband
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

ballots: (8, 8)
race24: (8, 11)
race20: (8, 11)
age: (8, 11)
gender: (8, 5)
education: (364, 5)
population: (455, 5)
poverty: (175, 5)
income: (707, 5)
broadband: (66, 4)


In [108]:
for name, df in datasets.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


BALLOTS COLUMNS
['County', 'Registered Voters', 'Total Ballots Cast', 'Democrat Ballots', 'Republican Ballots ', 'Absentee Ballots', 'Turnout as a Percentage of Registered Voters', 'Absentee as Percentage of Total Ballots']

RACE24 COLUMNS
['County', 'Total Ballots', 'Total Asian', 'Total American Indian', 'Total Black', 'Total Federally Registered', 'Total Hispanic', 'Total Korean', 'Total Other', 'Total White', 'Total Not Identified']

RACE20 COLUMNS
['County', 'Total Ballots', 'Total Asian', 'Total American Indian', 'Total Black', 'Total Federally Registered', 'Total Hispanic', 'Total Korean', 'Total Other', 'Total White', 'Total Not Identified']

AGE COLUMNS
['County', 'Total Ballots Cast', 'Age 18 - 29', 'Age 30 -39', 'Age 40 - 49', 'Age 50 - 59', 'Age 60 - 69', 'Age 70 - 79', 'Age 80 - 89', 'Age 90 -99', 'Age 100+']

GENDER COLUMNS
['County', 'Total Ballots Cast', 'Total Femal', 'Total Male', 'Total Not Identified']

EDUCATION COLUMNS
['FIPS Code', 'State', 'Area name', 'Attribu

In [109]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

ballots = clean_column_names(ballots)
race_2024 = clean_column_names(race_2024)
race_2020 = clean_column_names(race_2020)
age = clean_column_names(age)
gender = clean_column_names(gender)
education = clean_column_names(education)
population = clean_column_names(population)
poverty = clean_column_names(poverty)
income = clean_column_names(income)
broadband = clean_column_names(broadband)

In [110]:
for name, df in {
    "ballots": ballots,
    "race24": race_2024,
    "race20": race_2020,
    "age": age,
    "gender": gender,
    "education": education,
    "population": population,
    "poverty": poverty,
    "income": income,
    "broadband": broadband
    
}.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


BALLOTS COLUMNS
['county', 'registered_voters', 'total_ballots_cast', 'democrat_ballots', 'republican_ballots', 'absentee_ballots', 'turnout_as_a_percentage_of_registered_voters', 'absentee_as_percentage_of_total_ballots']

RACE24 COLUMNS
['county', 'total_ballots', 'total_asian', 'total_american_indian', 'total_black', 'total_federally_registered', 'total_hispanic', 'total_korean', 'total_other', 'total_white', 'total_not_identified']

RACE20 COLUMNS
['county', 'total_ballots', 'total_asian', 'total_american_indian', 'total_black', 'total_federally_registered', 'total_hispanic', 'total_korean', 'total_other', 'total_white', 'total_not_identified']

AGE COLUMNS
['county', 'total_ballots_cast', 'age_18__29', 'age_30_39', 'age_40__49', 'age_50__59', 'age_60__69', 'age_70__79', 'age_80__89', 'age_90_99', 'age_100']

GENDER COLUMNS
['county', 'total_ballots_cast', 'total_femal', 'total_male', 'total_not_identified']

EDUCATION COLUMNS
['fips_code', 'state', 'area_name', 'attribute', 'valu

In [111]:
def standardize_county(series):
    return (
        series.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(", AL", "", regex=False)
        .str.replace(" COUNTY", "", regex=False)
    )

ballots["county"] = standardize_county(ballots["county"])
race_2024["county"] = standardize_county(race_2024["county"])
race_2020["county"] = standardize_county(race_2020["county"])
age["county"] = standardize_county(age["county"])
gender["county"] = standardize_county(gender["county"])

education["county"] = standardize_county(education["area_name"])
population["county"] = standardize_county(population["area_name"])
poverty["county"] = standardize_county(poverty["area_name"])
income["county"] = standardize_county(income["area_name"])

broadband["county"] = standardize_county(broadband["county"])

In [112]:
def pivot_usda(df, county_col="county", attr_col="attribute", value_col="value"):
    wide = df.pivot_table(
        index=county_col,
        columns=attr_col,
        values=value_col,
        aggfunc="first"
    ).reset_index()
    wide.columns.name = None
    return wide

education_wide = pivot_usda(education)
population_wide = pivot_usda(population)
poverty_wide = pivot_usda(poverty)
income_wide = pivot_usda(income)

In [113]:
for name, df in {
    "education_wide": education_wide,
    "population_wide": population_wide,
    "poverty_wide": poverty_wide
}.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


EDUCATION_WIDE COLUMNS
['county', '2013 Rural-urban Continuum Code', '2013 Urban Influence Code', '2023 Rural-urban Continuum Code', '2024 Urban Influence Code', "Bachelor's degree or higher, 1990", "Bachelor's degree or higher, 2000", "Bachelor's degree or higher, 2008-12", "Bachelor's degree or higher, 2019-23", 'Four years of college or higher, 1970', 'Four years of college or higher, 1980', 'High school diploma only, 1970', 'High school diploma only, 1980', 'High school graduate (or equivalency), 1990', 'High school graduate (or equivalency), 2000', 'High school graduate (or equivalency), 2008-12', 'High school graduate (or equivalency), 2019-23', 'Less than a high school diploma, 1970', 'Less than a high school diploma, 1980', 'Less than high school graduate, 1990', 'Less than high school graduate, 2000', 'Less than high school graduate, 2008-12', 'Less than high school graduate, 2019-23', 'Percent of adults completing four years of college or higher, 1970', 'Percent of adults co

# Voter Turnout

In [114]:
ballots_sub = ballots[[
    "county",
    "registered_voters",
    "total_ballots_cast",
    "turnout_as_a_percentage_of_registered_voters",
    "absentee_as_percentage_of_total_ballots"
]].copy()

ballots_sub["turnout_rate"] = ballots_sub["turnout_as_a_percentage_of_registered_voters"] / 100
ballots_sub["absentee_rate"] = ballots_sub["absentee_as_percentage_of_total_ballots"] / 100

ballots_sub.head()

,county,registered_voters,total_ballots_cast,turnout_as_a_percentage_of_registered_voters,absentee_as_percentage_of_total_ballots,turnout_rate,absentee_rate
0,BALDWIN,207643,122542,59.02,6.34,0.5902,0.0634
1,CLARKE,19143,11993,62.65,6.04,0.6265,0.0604
2,CONECUH,9820,6105,62.17,7.85,0.6217,0.0785
3,ESCAMBIA,28268,15009,53.10,4.77,0.5310,0.0477
4,MOBILE,322535,176019,54.57,4.89,0.5457,0.0489


# Education

In [115]:
education_sub = education_wide[[
    "county",
    "Percent of adults who are not high school graduates, 2019-23",
    "Percent of adults with a bachelor's degree or higher, 2019-23"
]].copy()

education_sub = education_sub.rename(columns={
    "Percent of adults who are not high school graduates, 2019-23": "pct_less_hs",
    "Percent of adults with a bachelor's degree or higher, 2019-23": "pct_bachelors"
})

education_sub.head()

,county,pct_less_hs,pct_bachelors
0,BALDWIN,8.268600,32.797637
1,CLARKE,14.731304,15.426531
2,CONECUH,12.513843,13.129076
3,ESCAMBIA,17.467352,12.490686
4,MOBILE,10.991693,25.150224


# Population

In [116]:
population_sub = population_wide[[
    "county",
    "POP_ESTIMATE_2023"
]].copy()

population_sub = population_sub.rename(columns={
    "POP_ESTIMATE_2023": "population_2023"
})

population_sub.head()

,county,population_2023
0,BALDWIN,253507.0
1,CLARKE,22337.0
2,CONECUH,11174.0
3,ESCAMBIA,36558.0
4,MOBILE,411640.0


# Rural

In [117]:
population_features = population_wide[[
    "county",
    "Rural_Urban_Continuum_Code_2023"
]].copy()

population_features = population_features.rename(columns={
    "Rural_Urban_Continuum_Code_2023": "rural_urban_code"
})

population_features.head()

,county,rural_urban_code
0,BALDWIN,3.0
1,CLARKE,9.0
2,CONECUH,9.0
3,ESCAMBIA,6.0
4,MOBILE,2.0


# Poverty

In [118]:
poverty_sub = poverty_wide[[
    "county",
    "PCTPOVALL_2023"
]].copy()

poverty_sub = poverty_sub.rename(columns={
    "PCTPOVALL_2023": "poverty_rate"
})

poverty_sub.head()

,county,poverty_rate
0,BALDWIN,10.0
1,CLARKE,19.0
2,CONECUH,26.5
3,ESCAMBIA,21.3
4,MOBILE,16.3


# Race

In [119]:
# 2024
race_2024["non_white_share_2024"] = (
    race_2024["total_ballots"] - race_2024["total_white"]
) / race_2024["total_ballots"]

race_2024_sub = race_2024[[
    "county",
    "non_white_share_2024"
]].copy()

# 2020
race_2020["non_white_share_2020"] = (
    race_2020["total_ballots"] - race_2020["total_white"]
) / race_2020["total_ballots"]

race_2020_sub = race_2020[[
    "county",
    "non_white_share_2020"
]].copy()

In [120]:
race_combined = race_2024_sub.merge(
    race_2020_sub,
    on="county",
    how="inner"
)

race_combined.head()

,county,non_white_share_2024,non_white_share_2020
0,BALDWIN,0.077836,0.088134
1,CLARKE,0.405595,0.421104
2,CONECUH,0.405259,0.458399
3,ESCAMBIA,0.251116,0.292079
4,MOBILE,0.348480,0.363349


In [121]:
race_combined["non_white_share"] = (
    race_combined["non_white_share_2024"] +
    race_combined["non_white_share_2020"]
) / 2

race_sub = race_combined[[
    "county",
    "non_white_share"
]].copy()

race_sub.head()

,county,non_white_share
0,BALDWIN,0.082985
1,CLARKE,0.413350
2,CONECUH,0.431829
3,ESCAMBIA,0.271598
4,MOBILE,0.355915


# Unemployment

In [122]:
unemployment_sub = income_wide[[
    "county",
    "Unemployment_rate_2023"
]].copy()

unemployment_sub = unemployment_sub.rename(columns={
    "Unemployment_rate_2023": "unemployment_rate"
})

unemployment_sub.head()

,county,unemployment_rate
0,BALDWIN,2.3
1,CLARKE,5.0
2,CONECUH,3.5
3,ESCAMBIA,3.1
4,MOBILE,3.1


# Median Household Income

In [123]:
income_sub = income_wide[[
    "county",
    "Median_Household_Income_2022"
]].copy()

income_sub = income_sub.rename(columns={
    "Median_Household_Income_2022": "median_income"
})

income_sub.head()

,county,median_income
0,BALDWIN,71704.0
1,CLARKE,44906.0
2,CONECUH,36106.0
3,ESCAMBIA,47792.0
4,MOBILE,54315.0


# Broadband Access

In [124]:
broadband_sub = broadband[[
    "county",
    "_households_with_broadband_access"
]].copy()

broadband_sub = broadband_sub.rename(columns={
    "_households_with_broadband_access": "broadband_households"
})

broadband_sub.head()

,county,broadband_households
0,BALDWIN,"89,266"
1,CLARKE,"6,206"
2,CONECUH,"3,211"
3,ESCAMBIA,"10,114"
4,MOBILE,"139,490"


# Merging

In [125]:
merged = ballots_sub.merge(population_sub, on="county", how="left")
merged = merged.merge(poverty_sub, on="county", how="left")
merged = merged.merge(education_sub, on="county", how="left")
merged = merged.merge(unemployment_sub, on="county", how="left")
merged = merged.merge(income_sub, on="county", how="left")
merged = merged.merge(race_sub, on="county", how="left")
# removed age for now due to lack of important signal
merged = merged.merge(population_features, on="county", how="left")
merged = merged.merge(broadband_sub, on="county", how="left")
merged.head()

,county,registered_voters,total_ballots_cast,turnout_as_a_percentage_of_registered_voters,absentee_as_percentage_of_total_ballots,turnout_rate,absentee_rate,population_2023,poverty_rate,pct_less_hs,pct_bachelors,unemployment_rate,median_income,non_white_share,rural_urban_code,broadband_households
0,BALDWIN,207643,122542,59.02,6.34,0.5902,0.0634,253507.0,10.0,8.268600,32.797637,2.3,71704.0,0.082985,3.0,"89,266"
1,CLARKE,19143,11993,62.65,6.04,0.6265,0.0604,22337.0,19.0,14.731304,15.426531,5.0,44906.0,0.413350,9.0,"6,206"
2,CONECUH,9820,6105,62.17,7.85,0.6217,0.0785,11174.0,26.5,12.513843,13.129076,3.5,36106.0,0.431829,9.0,"3,211"
3,ESCAMBIA,28268,15009,53.10,4.77,0.5310,0.0477,36558.0,21.3,17.467352,12.490686,3.1,47792.0,0.271598,6.0,"10,114"
4,MOBILE,322535,176019,54.57,4.89,0.5457,0.0489,411640.0,16.3,10.991693,25.150224,3.1,54315.0,0.355915,2.0,"139,490"


In [126]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 16 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   county                                        8 non-null      object 
 1   registered_voters                             8 non-null      int64  
 2   total_ballots_cast                            8 non-null      int64  
 3   turnout_as_a_percentage_of_registered_voters  8 non-null      float64
 4   absentee_as_percentage_of_total_ballots       8 non-null      float64
 5   turnout_rate                                  8 non-null      float64
 6   absentee_rate                                 8 non-null      float64
 7   population_2023                               7 non-null      float64
 8   poverty_rate                                  7 non-null      float64
 9   pct_less_hs                                   7 non-null      float64

In [127]:
merged.isna().sum().sort_values(ascending=False)

population_2023                                 1
broadband_households                            1
rural_urban_code                                1
median_income                                   1
unemployment_rate                               1
pct_bachelors                                   1
pct_less_hs                                     1
poverty_rate                                    1
absentee_rate                                   0
turnout_rate                                    0
absentee_as_percentage_of_total_ballots         0
turnout_as_a_percentage_of_registered_voters    0
registered_voters                               0
total_ballots_cast                              0
county                                          0
non_white_share                                 0
dtype: int64

In [128]:
merged.describe()

,registered_voters,total_ballots_cast,turnout_as_a_percentage_of_registered_voters,absentee_as_percentage_of_total_ballots,turnout_rate,absentee_rate,population_2023,poverty_rate,pct_less_hs,pct_bachelors,unemployment_rate,median_income,non_white_share,rural_urban_code
count,8.000000e+00,8.000000e+00,8.000000,8.000000,8.000000,8.000000,7.000000,7.00000,7.000000,7.000000,7.000000,7.000000,8.000000,7.000000
mean,5.599346e+05,3.278660e+05,59.028750,6.065000,0.590287,0.060650,109923.857143,18.70000,13.096151,18.079458,3.414286,50872.285714,0.307679,6.428571
std,1.339251e+06,7.885548e+05,3.507492,1.210808,0.035075,0.012108,159000.110167,4.98832,3.262040,7.938253,0.949436,11021.269054,0.114484,2.878492
min,9.820000e+03,6.105000e+03,53.100000,4.770000,0.531000,0.047700,11174.000000,10.00000,8.268600,11.357915,2.300000,36106.000000,0.082985,2.000000
25%,1.571025e+04,9.517250e+03,57.780000,5.175000,0.577800,0.051750,17125.500000,17.50000,11.153325,12.809881,2.850000,46202.000000,0.261054,4.500000
50%,2.370550e+04,1.350100e+04,59.660000,5.795000,0.596600,0.057950,22337.000000,19.00000,12.513843,15.426531,3.100000,47792.000000,0.313756,8.000000
75%,2.363660e+05,1.359112e+05,61.720000,6.707500,0.617200,0.067075,145032.500000,20.20000,15.558307,20.677179,3.900000,54050.000000,0.393782,8.500000
max,3.861929e+06,2.272911e+06,62.650000,7.850000,0.626500,0.078500,411640.000000,26.50000,17.467352,32.797637,5.000000,71704.000000,0.431829,9.000000


In [129]:
# Drop duplicate percentage columns
merged = merged.drop(columns=[
    "turnout_as_a_percentage_of_registered_voters",
    "absentee_as_percentage_of_total_ballots"
])

merged.head()

,county,registered_voters,total_ballots_cast,turnout_rate,absentee_rate,population_2023,poverty_rate,pct_less_hs,pct_bachelors,unemployment_rate,median_income,non_white_share,rural_urban_code,broadband_households
0,BALDWIN,207643,122542,0.5902,0.0634,253507.0,10.0,8.268600,32.797637,2.3,71704.0,0.082985,3.0,"89,266"
1,CLARKE,19143,11993,0.6265,0.0604,22337.0,19.0,14.731304,15.426531,5.0,44906.0,0.413350,9.0,"6,206"
2,CONECUH,9820,6105,0.6217,0.0785,11174.0,26.5,12.513843,13.129076,3.5,36106.0,0.431829,9.0,"3,211"
3,ESCAMBIA,28268,15009,0.5310,0.0477,36558.0,21.3,17.467352,12.490686,3.1,47792.0,0.271598,6.0,"10,114"
4,MOBILE,322535,176019,0.5457,0.0489,411640.0,16.3,10.991693,25.150224,3.1,54315.0,0.355915,2.0,"139,490"


In [130]:
OUTPUT_DIR = BASE_DIR / "data" / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

merged.to_csv(OUTPUT_DIR / "gc_cpri_model_input.csv", index=False)